<a href="https://colab.research.google.com/github/mic006016/geo-referencing-ai-pipeline/blob/main/GeoAI_YOLOv8_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import shutil

# 1. 구글 드라이브 연결
drive.mount('/content/drive')

# 2. 드라이브의 zip 파일을 코랩의 로컬 가상 디스크(/content)로 복사
zip_path = "/content/drive/MyDrive/yolo_dataset.zip"
local_zip_path = "/content/yolo_dataset.zip"

print("데이터 복사 중...")
shutil.copy(zip_path, local_zip_path)
print("복사 완료!")

# 3. 로컬에서 압축 해제 (-q 옵션으로 출력 생략하여 브라우저 렉 방지)
!unzip -q {local_zip_path} -d /content/yolo_dataset
print("압축 해제 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
데이터 복사 중...
복사 완료!
압축 해제 완료!


In [ ]:
# Ultralytics 라이브러리 설치
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 61.9/235.7 GB disk)


In [ ]:
from ultralytics import YOLO

# 1. 모델 로드
model = YOLO('yolov8s.pt')

# 2. 전이 학습 시작
results = model.train(
    data='/content/yolo_dataset/dataset.yaml',
    epochs=50,
    imgsz=512,
    batch=64,
    device=0,              # 0번 GPU 사용
    workers=8,             # 데이터 로딩에 사용할 CPU 워커 수
    amp=True,              # 자동 혼합 정밀도 (연산 속도 가속)
    project='geo_ai',      # 결과물이 저장될 폴더명
    name='land_cover_v1',
    save=True,             # 매 에포크마다 가중치 저장
    patience=10,           # 10 에포크 동안 성능 향상이 없으면 조기 종료
)

Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=land_cover_v1

In [ ]:
#3. 테스트셋 평가
metrics = model.val(
    data='/content/yolo_dataset/dataset.yaml',
    split='test',          # 훈련에 관여하지 않은 테스트 데이터 지정
    project='geo_ai',
    name='land_cover_v1_test'
)

Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2270.2±649.1 MB/s, size: 109.2 KB)
val: Scanning /content/yolo_dataset/test/labels... 4465 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4465/4465 1.4Kit/s 3.2s
val: New cache created: /content/yolo_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 280/280 10.1it/s 27.7s
                   all       4465      62734      0.833      0.793      0.846      0.697
              Building       2971      22477      0.813      0.742      0.821      0.638
            GreenHouse       2182      11392      0.798       0.84      0.846      0.706
                  Road       1543       2249      0.872      0.855      0.899      0.783
             RicePaddy       2803      14740      0.9

In [ ]:
from google.colab import files

# 1. geo_ai 폴더를 results_v1.zip으로 압축 (-q 옵션으로 로그 생략)
!zip -r -q /content/results_v1.zip /content/runs/detect/geo_ai/land_cover_v1_test

print("✅ 압축 완료! PC로 다운로드를 시작합니다.")

# 2. 브라우저를 통해 내 로컬 PC로 다운로드
files.download('/content/results_v1.zip')

✅ 압축 완료! PC로 다운로드를 시작합니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import cv2
import shutil
from ultralytics import YOLO

# ==========================================
# 1. 환경, 클래스 ID 및 추출 제한 설정
# ==========================================
TEST_IMG_DIR = '/content/yolo_dataset/test/images'
TEST_LABEL_DIR = '/content/yolo_dataset/test/labels'
OUTPUT_DIR = '/content/error'

# 폴더가 이미 있다면 꼬이지 않게 삭제 후 재생성 (선택적 초기화)
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

RICEPADDY_CLASS_ID = 3  # 논
FIELD_CLASS_ID = 4      # 밭

# 💡 딱 20장만 뽑기 위한 제한 설정
MAX_SAMPLES = 20
sample_count = 0

model = YOLO('/content/runs/detect/geo_ai/land_cover_v1-2/weights/best.pt')

print(f"🔍 오분류(밭 -> 논) 이미지 딱 {MAX_SAMPLES}장만 추출 시작...")

# ==========================================
# 2. 테스트셋 순회 및 샘플링
# ==========================================
for img_name in os.listdir(TEST_IMG_DIR):
    if not img_name.endswith(('.jpg', '.png', '.jpeg')):
        continue

    img_path = os.path.join(TEST_IMG_DIR, img_name)
    label_path = os.path.join(TEST_LABEL_DIR, os.path.splitext(img_name)[0] + '.txt')

    # 조건 A: 정답지에 '밭(Field)' 존재 여부 확인
    has_field_in_gt = False
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                cls_id = int(line.strip().split()[0])
                if cls_id == FIELD_CLASS_ID:
                    has_field_in_gt = True
                    break

    if not has_field_in_gt:
        continue

    # 조건 B: 모델이 '논(RicePaddy)'으로 예측했는지 여부 확인
    results = model.predict(source=img_path, conf=0.25, verbose=False)

    has_ricepaddy_in_pred = False
    for box in results[0].boxes:
        pred_cls_id = int(box.cls[0].item())
        if pred_cls_id == RICEPADDY_CLASS_ID:
            has_ricepaddy_in_pred = True
            break

    # ==========================================
    # 3. 오분류 확인 시 추출 및 카운트
    # ==========================================
    if has_field_in_gt and has_ricepaddy_in_pred:
        sample_count += 1
        print(f"🚨 오분류 샘플 수집 중: {sample_count}/{MAX_SAMPLES} ({img_name})")

        # 예측 바운딩 박스가 포함된 이미지 저장
        result_img = results[0].plot()
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"pred_{img_name}"), result_img)

        # Feature Map (히트맵) 시각화 및 저장
        model.predict(
            source=img_path,
            conf=0.25,
            visualize=True,
            save=True,
            project=OUTPUT_DIR,
            name=f"heatmap_{os.path.splitext(img_name)[0]}"
        )

        # 지정한 개수를 채우면 반복문 강제 종료 (무한 루프 및 과부하 방지)
        if sample_count >= MAX_SAMPLES:
            print(f"🛑 목표 샘플 {MAX_SAMPLES}장 추출 완료. 작업을 중단합니다.")
            break

print(f"✅ 검증 샘플 추출 완료! 폴더 경로: '{OUTPUT_DIR}'")

🔍 오분류(밭 -> 논) 이미지 딱 20장만 추출 시작...
🚨 오분류 샘플 수집 중: 1/20 (LC_GG_AP25_37608081_020_2021.jpg)
Saving /content/error/heatmap_LC_GG_AP25_37608081_020_2021/LC_GG_AP25_37608081_020_2021/stage0_Conv_features.png... (32/32)
Saving /content/error/heatmap_LC_GG_AP25_37608081_020_2021/LC_GG_AP25_37608081_020_2021/stage1_Conv_features.png... (32/64)
Saving /content/error/heatmap_LC_GG_AP25_37608081_020_2021/LC_GG_AP25_37608081_020_2021/stage2_C2f_features.png... (32/64)
Saving /content/error/heatmap_LC_GG_AP25_37608081_020_2021/LC_GG_AP25_37608081_020_2021/stage3_Conv_features.png... (32/128)
Saving /content/error/heatmap_LC_GG_AP25_37608081_020_2021/LC_GG_AP25_37608081_020_2021/stage4_C2f_features.png... (32/128)
Saving /content/error/heatmap_LC_GG_AP25_37608081_020_2021/LC_GG_AP25_37608081_020_2021/stage5_Conv_features.png... (32/256)
Saving /content/error/heatmap_LC_GG_AP25_37608081_020_2021/LC_GG_AP25_37608081_020_2021/stage6_C2f_features.png... (32/256)
Saving /content/error/heatmap_LC_GG_AP25_3

In [ ]:
# 밭 -> 논 예측 파일 무작위 20장
!zip -r -q /content/error.zip /content/error
print("✅ 압축 완료! PC로 다운로드를 시작합니다.")

shutil.copy('/content/error.zip', '/content/drive/MyDrive/error.zip')
print("✅ 구글 드라이브로 복사 완료!")

✅ 압축 완료! PC로 다운로드를 시작합니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>